In [ ]:
!pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 41.7 MB/s eta 0:00:00


In [ ]:
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base_folder = "/content/drive/MyDrive/GEE_Exports"
for f in os.listdir(base_folder):
    print(f)

khanchengyao_x_train_image.tif
khanchengyao_y_train.tif
langpo_x_train_image.tif
langpo_y_train.tif
barlacha_y_train.tif
DEM
gepang_x_train_image.tif
gepang_y_train.tif
fanchan_x_train_image.tif
fanchan_y_train.tif
barlacha_x_train_image.tif
Shapefiles
fanchan_x_train_enhanced.tif
gepang_x_train_enhanced.tif
karzok_x_train_enhanced.tif
khanchengyao_x_train_enhanced.tif
langpo_x_train_enhanced.tif
vasuki_y_train.tif
vasuki_x_train_image.tif
samundraTapu_x_train_image.tif
samundraTapu_y_train.tif
samundraTapu_x_train_enhanced.tif
random_forest_tree.png
barlacha_x_train_enhanced.tif
glacial_lake_dataset (1).csv
Predicted
glacial_lake_dataset_polygon.csv
karzok_x_train_image.tif
karzok_y_train.tif
karzok_predicted2.tif
langpo_predicted2.tif
glacial_lake_dataset_with_DEM.csv
vasuki_x_train_enhanced.tif
neelkanth_x_train_image.tif
beas_x_train_image.tif
beas_y_train.tif
neelkanth_y_train.tif
barlacha_predicted.tif
khanchengyao_predicted.tif
gepang_predicted.tif
fanchan_predicted.tif
langpo_p

In [ ]:
x_train_path = "/content/drive/MyDrive/GEE_Exports/hangu_x_train_image.tif"
dem_path     = "/content/drive/MyDrive/GEE_Exports/DEM/hangu.tif"
out_path     = "/content/drive/MyDrive/GEE_Exports/hangu_x_train_enhanced.tif"

In [ ]:
with rasterio.open(x_train_path) as src:
    sentinel = src.read()   # shape: (bands, H, W)
    meta = src.meta.copy()
    transform = src.transform
    crs = src.crs

print("Sentinel shape (bands, H, W):", sentinel.shape)

# Bands (assuming order: B2, B3, B4, B8, B11, B12)
blue, green, red, nir, swir1, swir2 = sentinel[:6]

Sentinel shape (bands, H, W): (6, 35, 46)


In [ ]:
ndwi = (green - nir) / (green + nir + 1e-6)

# NDSI
ndsi = (green - swir1) / (green + swir1 + 1e-6)

In [ ]:
with rasterio.open(dem_path) as src:
    dem = src.read(1).astype("float32")
    dem_transform = src.transform
    dem_crs = src.crs

print("DEM shape:", dem.shape)

DEM shape: (54, 69)


In [ ]:
if dem.shape != (sentinel.shape[1], sentinel.shape[2]) or dem_crs != crs:
    dem_resampled = np.empty((sentinel.shape[1], sentinel.shape[2]), dtype=np.float32)
    reproject(
        source=dem,
        destination=dem_resampled,
        src_transform=dem_transform,
        src_crs=dem_crs,
        dst_transform=transform,
        dst_crs=crs,
        resampling=Resampling.bilinear
    )
    dem = dem_resampled

In [ ]:
# Pixel size
xres = transform[0]
yres = -transform[4]

# Gradients
dy, dx = np.gradient(dem, yres, xres)
slope = np.arctan(np.sqrt(dx**2 + dy**2))  # slope in radians

# Aspect
aspect = np.arctan2(-dx, dy)

# Hillshade (simple Lambertian model)
azimuth = 315.0 * np.pi / 180.0
altitude = 45.0 * np.pi / 180.0
hillshade = (np.sin(altitude) * np.cos(slope) +
             np.cos(altitude) * np.sin(slope) * np.cos(azimuth - aspect))
hillshade = np.clip(hillshade, 0, 1)

In [ ]:
enhanced = np.vstack([
    sentinel,           # original Sentinel bands
    ndwi[np.newaxis],   # NDWI
    ndsi[np.newaxis],   # NDSI
    slope[np.newaxis],  # slope
    hillshade[np.newaxis], # hillshade
    dem[np.newaxis]     # elevation
])

print("Final enhanced shape:", enhanced.shape)

Final enhanced shape: (11, 35, 46)


In [ ]:
meta.update({
    "count": enhanced.shape[0],
    "dtype": "float32"
})

with rasterio.open(out_path, "w", **meta) as dst:
    dst.write(enhanced.astype("float32"))

print("Enhanced x_train saved at:", out_path)

Enhanced x_train saved at: /content/drive/MyDrive/GEE_Exports/hangu_x_train_enhanced.tif


In [ ]:
dem_path

'/content/drive/MyDrive/GEE_Exports/DEM/hangu.tif'